# PhysAR-IDS -- NB6 (v6): Attack 3 -- an adaptive attacker that targets both layers

**Reviewer point (issue 2).** *"The adaptive attacker ignores the autoencoder gate. A properly adaptive attacker who knows the whole pipeline would also try to keep reconstruction error under tau (Tramer et al., 2020). Either test an attacker that targets both layers, or tone down the worst-case claim."* This notebook tests it.

**The pipeline.** Layer 1 = the adversarially trained XGBoost (NB5 defended model, and the NB5b white-box-trained "robust" model). Layer 2 = an autoencoder gate trained on benign training traffic only: five autoencoders (5-fold), threshold tau = 95th percentile of the pooled out-of-fold benign reconstruction error, a row is flagged when the ensemble-mean error exceeds tau. An attack row **evades the pipeline only if Layer 1 calls it benign AND the gate does not flag it.** (The gate is trained here, per seed, and saved as `ae_gate_v6_seed{s}.pt`; NB7 can reload it.)

**Attackers, from least to most informed (all on the 60 attacker-controlled cyber features, budgets 0.1 / 0.3 / 1.0 sigma, test rows only):**
1. *Non-adaptive:* PGD from NB3's surrogate and the saved NB4 PPO agents (made against the standard classifier, gate-unaware), evaluated against the pipeline.
2. *Layer-1-adaptive, gate-unaware* (what NB5 / NB5b measured): the white-box tree-greedy attack against Layer 1 only; we then ask how much of what evades Layer 1 the gate catches.
3. ***Fully adaptive white-box*:** the same tree-greedy search, but it maximises the joint score `min(margin gap, GATE_SCALE * (tau - error) / tau)` using the real classifier scores and the real gate, so it stops only when **both** conditions hold. Every decision reported is made by the CPU classifier and the gate.
4. ***Fully adaptive black-box*:** a recurrent PPO agent retrained (NB4 recipe, 50k steps) against the pipeline's hard-label verdict (benign-and-unflagged = success).

**What this answers.** (i) the gate on its own (detection of clean and attacked rows; the "no result for the gate on its own" comment); (ii) whether the gate adds protection beyond Layer 1; (iii) whether an attacker that knows the gate removes that protection. As before the white-box search is greedy, so evasion rates are **lower bounds** on what an optimal attacker could do; the number of rows still improving at the round limit is reported.

**Needs:** `xgb_robust_v6_seed{s}.json` (NB5b), `xgb_defended_v6_seed{s}.json` (NB5), `surrogate_mlp_v6_seed{s}.pt` (NB3), the saved NB4 agents, and the data files. **Runtime:** several hours (about 1.5-2 h per seed); results are saved per seed.

## 0. Setup

In [ ]:
import os, sys, random, time, json, hashlib, shutil, platform, warnings, copy
import numpy as np, pandas as pd, joblib
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import xgboost as xgb
from scipy import stats
from sklearn.metrics import f1_score
import matplotlib; import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
try:
    from sb3_contrib import RecurrentPPO
    from stable_baselines3.common.callbacks import CheckpointCallback
except ImportError as e:
    raise ImportError('Install in a terminal (not in a cell): pip install stable-baselines3 sb3-contrib gymnasium --no-deps') from e
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = False

# ---- PPO / environment settings (unchanged from the reference design) ----
EPS_BOUND = 0.3; MAX_STEPS = 10
EVASION_REWARD = 10.0; STEP_PENALTY_SCALE = 0.01; TIMEOUT_PENALTY = -1.0
TOTAL_TIMESTEPS = 50000; CKPT_FREQ = 5000
EPS_TOTAL_BUDGET = 0.1
CLIP_BUDGETS = {'1.0': 1.0, 'unconstrained': None}     # budgets at which agents are RETRAINED with range clipping (stage B)
SAVED_AGENTS = {'0.1': ('ppo_attack2_budgeted_{v}_seed{s}', 0.1), '0.2': ('ppo_attack2_budget0p2_{v}_seed{s}', 0.2), '0.3': ('ppo_attack2_budget0p3_{v}_seed{s}', 0.3),
                '0.5': ('ppo_attack2_budget0p5_{v}_seed{s}', 0.5), '1.0': ('ppo_attack2_budget1p0_{v}_seed{s}', 1.0), 'unconstrained': ('ppo_attack2_unconstrained_{v}_seed{s}', None)}
EVAL_CHUNK = 8192


# ---- Defence 1 (adversarial training) settings ----
PGD_EPS = 0.1; PGD_ITERS = 10                       # as NB3 / NB3b (all 93 features, alpha = eps / 4)
TRAIN_PPO_TAGS = ['0.1', '0.3']                     # PPO agents (NB4 / NB4b, trained against the STANDARD model) whose examples go into the training set
EVAL_PPO_TAGS = ['0.1', '0.3', '0.5', '1.0', 'unconstrained']   # budgets 0.5 / 1.0 / unconstrained were never seen in training
N_PPO_TRAIN_PER_CLASS = 8000
AUG_RATIO = 1.0                                     # adversarial training rows per clean row (the paper's 1:1)
# ---- white-box (tree-greedy, NB3c / NB3d) attack against both models ----
K_CAND = 12; MAX_ITERS = 60; CHUNK = 20000; BLOCK = 300_000
WB_BUDGETS = {'0.1': 0.1, '0.3': 0.3, '0.5': 0.5, '1.0': 1.0}
N_WB_PER_CLASS = 2000                               # attack rows per class searched (same rows for every seed, model and budget)


# ---- NB6: Attack 3 -- adaptive attacker against the WHOLE pipeline (Layer 1 classifier + Layer 2 autoencoder gate) ----
LAYER1 = {'rob': 'xgb_robust_{v}_seed{s}.json', 'def': 'xgb_defended_{v}_seed{s}.json'}      # NB5b robust model (primary) and NB5 defended model
AW_BUDGETS = {'0.1': 0.1, '0.3': 0.3, '1.0': 1.0}      # sigma, cyber features only (same attacker-controlled set as NB3c / NB5)
N_AW_PER_CLASS = 1000                                    # rows per attack class (a random subset of NB5's 2,000 per class)
MAX_ITERS_AWARE = 100                                    # rounds for the greedy search against the pipeline
GATE_SCALE = 1.0                                         # weight of the gate term in the joint score min(margin gap, GATE_SCALE * (tau - err) / tau)
N_FOLDS = 5; GATE_EPOCHS = 30; GATE_PCTL = 95; GATE_HIDDEN = 32; GATE_BOTTLENECK = 12; GATE_CYBER_BOOST = 1.0
RUN_PPO_AWARE = True; PPO_AWARE_MODELS = ['rob']; PPO_AWARE_BUDGETS = {'0.3': 0.3, '1.0': 1.0}      # black-box PPO retrained against the pipeline's hard-label verdict
PPO_UNAWARE_TAGS = ['0.3', '1.0', 'unconstrained']      # the saved NB4 agents (trained against the standard classifier, gate-unaware)

os.environ['PYTHONHASHSEED'] = '42'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
BACKEND_NAME = 'cpu(hist)' if _cpu else 'gpu(cuda,hist)'
print(f'torch {torch.__version__} ({DEVICE}) | xgboost {xgb.__version__} fit backend: {BACKEND_NAME}')
import sb3_contrib, stable_baselines3
print(f'sb3-contrib {sb3_contrib.__version__} | stable-baselines3 {stable_baselines3.__version__} | gymnasium {gym.__version__}')
print('Working directory:', os.getcwd())

## 1. Load v6 data

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]; N_CYBER = len(CYBER_IDX)
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)

atk_tr = y_train != 0; X_atk_train, y_atk_train = X_train[atk_tr], y_train[atk_tr]
atk_te = y_test != 0;  X_atk_test,  y_atk_test  = X_test[atk_te],  y_test[atk_te]
test_burst = pd.factorize(pd.MultiIndex.from_arrays([y_test, np.asarray(data['PLCID_test']), np.asarray(data['timestamp_test'])]))[0][atk_te]
FEAT_LO, FEAT_HI = X_train.min(0), X_train.max(0)
print(f'Features {N_FEAT} | attacker-controlled (cyber) {N_CYBER} | training pool {X_atk_train.shape} | evaluation rows {X_atk_test.shape} in {len(np.unique(test_burst))} attack bursts')
print('Training pool class counts:', np.bincount(y_atk_train)[1:], '| evaluation class counts:', np.bincount(y_atk_test)[1:])
nb2 = joblib.load(f'optuna_xgboost_results_{PIPELINE_VERSION}.pkl')
nb2_tuned = np.array([nb2['results']['tuned'][s]['test_f1'] for s in SEEDS if s in nb2['results']['tuned']])
print(f'NB2 tuned test F1 over seeds: {nb2_tuned.mean():.4f} +/- {nb2_tuned.std(ddof=1):.4f}')

## 2. Oracle, PPO environment (NB4 code)

In [ ]:
class Oracle:
    """Hard-label black-box oracle over the fitted XGBoost classifier."""
    def __init__(self, clf): self.clf = clf; self.n_queries = 0
    def predict(self, X):
        X = np.ascontiguousarray(X, dtype=np.float32); self.n_queries += len(X)
        return self.clf.predict(X)

def apply_action(x_cur, x_orig, action, eps_bound=EPS_BOUND, eps_total=None, range_clip=False):
    """In place on x_cur[:, CYBER_IDX]; returns the applied (post-clamp) perturbation."""
    a = np.clip(action, -eps_bound, eps_bound).astype(np.float32)
    old = x_cur[:, CYBER_IDX].copy(); prop = old + a
    if eps_total is not None:
        o = x_orig[:, CYBER_IDX]; prop = o + np.clip(prop - o, -eps_total, eps_total)
    if range_clip:   # never leave the observed training range [min, max] (a value already outside it may stay where it was, but may not go further out)
        prop = np.clip(prop, np.minimum(FEAT_LO[CYBER_IDX], x_orig[:, CYBER_IDX]), np.maximum(FEAT_HI[CYBER_IDX], x_orig[:, CYBER_IDX]))
    x_cur[:, CYBER_IDX] = prop
    return (prop - old).astype(np.float32)

def build_obs(x_cur, last_applied, last_pred, step):
    oh = np.zeros((len(x_cur), 6), dtype=np.float32); oh[np.arange(len(x_cur)), last_pred] = 1.0
    return np.concatenate([x_cur.astype(np.float32), last_applied.astype(np.float32), oh, np.full((len(x_cur), 1), step / MAX_STEPS, dtype=np.float32)], 1)

class IDSEvasionEnv(gym.Env):
    metadata = {'render_modes': []}
    def __init__(self, X_pool, y_pool, oracle, eps_bound=EPS_BOUND, max_steps=MAX_STEPS, seed=None, eps_total_budget=None, range_clip=False):
        super().__init__(); self.range_clip = range_clip
        self.X_pool, self.y_pool, self.oracle = X_pool, y_pool, oracle
        self.eps_bound, self.max_steps, self.eps_total_budget = eps_bound, max_steps, eps_total_budget
        self.rng = np.random.RandomState(seed)
        self.action_space = spaces.Box(low=-eps_bound, high=eps_bound, shape=(N_CYBER,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(N_FEAT + N_CYBER + 6 + 1,), dtype=np.float32)
    def _obs(self): return build_obs(self.x_current[None], self.last_applied[None], np.array([self.last_pred]), self.step_count)[0]
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        idx = self.rng.randint(0, len(self.X_pool))
        self.x_orig = self.X_pool[idx].copy(); self.x_current = self.x_orig.copy(); self.y_true = self.y_pool[idx]
        self.step_count = 0; self.last_applied = np.zeros(N_CYBER, dtype=np.float32)
        self.last_pred = int(self.oracle.predict(self.x_current[None])[0]); return self._obs(), {}
    def step(self, action):
        applied = apply_action(self.x_current[None], self.x_orig[None], np.asarray(action, dtype=np.float32)[None], self.eps_bound, self.eps_total_budget, self.range_clip)[0]
        self.last_applied = applied; self.step_count += 1
        self.last_pred = int(self.oracle.predict(self.x_current[None])[0]); pen = STEP_PENALTY_SCALE * float(np.sum(applied ** 2))
        terminated = truncated = False
        if self.last_pred == 0: reward = EVASION_REWARD - pen; terminated = True
        elif self.step_count >= self.max_steps: reward = TIMEOUT_PENALTY - pen; truncated = True
        else: reward = -pen
        return self._obs(), reward, terminated, truncated, {'evaded': self.last_pred == 0}

def run_batched(policy, X0, oracle, eps_total, mode='ppo', rng=None, range_clip=False):
    """Run one episode per row of X0 in parallel (10 steps max). mode: 'ppo' (deterministic policy) or 'random'."""
    N = len(X0); x_cur = X0.copy(); last_applied = np.zeros((N, N_CYBER), dtype=np.float32)
    last_pred = np.concatenate([oracle.predict(X0[i:i + 65536]) for i in range(0, N, 65536)]).astype(np.int64); clean_pred = last_pred.copy()
    active = np.ones(N, bool); evaded = np.zeros(N, bool); steps = np.zeros(N, np.int64)
    if mode == 'ppo': h = np.zeros((1, N, 64), np.float32); c = np.zeros((1, N, 64), np.float32)
    for step in range(MAX_STEPS):
        idx = np.where(active)[0]
        if len(idx) == 0: break
        obs = build_obs(x_cur[idx], last_applied[idx], last_pred[idx], step)
        if mode == 'ppo':
            act = np.zeros((len(idx), N_CYBER), np.float32)
            for s in range(0, len(idx), EVAL_CHUNK):
                sl = idx[s:s + EVAL_CHUNK]
                a, (hn, cn) = policy.predict(obs[s:s + EVAL_CHUNK], state=(h[:, sl], c[:, sl]), episode_start=np.full(len(sl), step == 0), deterministic=True)
                act[s:s + EVAL_CHUNK] = a; h[:, sl] = hn; c[:, sl] = cn
        else:
            act = rng.uniform(-EPS_BOUND, EPS_BOUND, size=(len(idx), N_CYBER)).astype(np.float32)
        sub = x_cur[idx]; applied = apply_action(sub, X0[idx], act, EPS_BOUND, eps_total, range_clip); x_cur[idx] = sub; last_applied[idx] = applied
        pred = np.concatenate([oracle.predict(sub[i:i + 65536]) for i in range(0, len(sub), 65536)]).astype(np.int64); last_pred[idx] = pred
        ev = pred == 0; evaded[idx[ev]] = True; steps[idx[ev]] = step + 1; active[idx[ev]] = False
    return {'evaded': evaded, 'steps': steps, 'x_final': x_cur, 'pred_final': last_pred, 'clean_pred': clean_pred}

def per_class(vec_bool):
    return np.array([vec_bool[y_atk_test == c].mean() for c in range(1, 6)])
def mean_steps(res):
    return np.array([res['steps'][(y_atk_test == c) & res['evaded']].mean() if ((y_atk_test == c) & res['evaded']).any() else np.nan for c in range(1, 6)])
def summarise(res):
    disp = np.abs(res['x_final'][:, CYBER_IDX] - X_atk_test[:, CYBER_IDX])
    _ex = lambda Z: np.maximum(np.maximum(FEAT_LO[CYBER_IDX] - Z[:, CYBER_IDX], Z[:, CYBER_IDX] - FEAT_HI[CYBER_IDX]), 0).max(1)
    exc = np.maximum(_ex(res['x_final']) - _ex(X_atk_test), 0)    # sigma ADDED beyond what the clean test row already had outside the training min/max
    return {'per_class_evasion': per_class(res['evaded']), 'overall_evasion': float(res['evaded'].mean()), 'mean_steps_to_evade': mean_steps(res),
            'mean_queries': float(1 + np.where(res['evaded'], res['steps'], MAX_STEPS).mean()),
            'attack_f1': float(f1_score(y_atk_test, res['pred_final'], average='weighted')),
            'max_abs_disp_mean': float(disp.max(1).mean()), 'max_abs_disp_max': float(disp.max()), 'mean_excess_beyond_train_range': float(exc.mean()), 'frac_rows_beyond_range_0.5sigma': float((exc > 0.5).mean()),
            'evaded': res['evaded'].copy()}
print('environment + evaluator defined; cyber features controlled:', N_CYBER)

## 3. Full-test metrics (NB4e)

In [ ]:
from sklearn.metrics import precision_score, recall_score
LAB = list(range(6))
def full_metrics(pred_atk, benign_pred):
    """Full-test metrics when the attack rows carry predictions `pred_atk` and benign rows keep `benign_pred`."""
    pred = np.empty(len(y_test), dtype=np.int64); pred[y_test == 0] = benign_pred; pred[atk_te] = pred_atk
    return {'weighted_f1': float(f1_score(y_test, pred, average='weighted')), 'macro_f1': float(f1_score(y_test, pred, average='macro')),
            'per_class_f1': f1_score(y_test, pred, average=None, labels=LAB), 'per_class_recall': recall_score(y_test, pred, average=None, labels=LAB),
            'benign_precision': float(precision_score(y_test, pred, labels=[0], average=None)[0]), 'attack_detected': float((pred_atk != 0).mean())}
print('full_metrics defined; full test rows', len(y_test), '| attack rows', int(atk_te.sum()))

## 4. White-box tree-greedy attack (NB3c / NB3d code)

In [ ]:
def get_thresholds(clf):
    """feature index -> sorted unique float32 split thresholds used anywhere in the ensemble."""
    df = clf.get_booster().trees_to_dataframe(); sp = df[df['Feature'] != 'Leaf']
    return {int(str(f).lstrip('f')): np.unique(g['Split'].to_numpy(dtype=np.float32)) for f, g in sp.groupby('Feature')}

class Margin:
    """Raw per-class scores of a fitted classifier (the white-box access). Scores in blocks and, if the GPU runs out of memory, halves the block and retries."""
    def __init__(self, clf): self.b = clf.get_booster(); self.bs = BLOCK
    def _pred(self, X):
        while True:
            try:
                return [np.asarray(self.b.inplace_predict(np.ascontiguousarray(X[s:s + self.bs], dtype=np.float32), predict_type='margin', validate_features=False)) for s in range(0, len(X), self.bs)]
            except xgb.core.XGBoostError as e:
                if 'out of memory' not in str(e).lower() or self.bs <= 20000: raise
                self.bs = max(20000, self.bs // 2); print(f'      GPU out of memory -> scoring in blocks of {self.bs:,} rows', flush=True)
    def __call__(self, X):
        out = self._pred(X); return np.concatenate(out) if len(out) > 1 else out[0]
def gap_of(z): return (z[:, 0] - z[:, 1:].max(1)).astype(np.float64)       # > 0  <=>  benign is the arg-max class

def interval(o, j, B):
    """allowed interval of feature j for rows with original values o: L-inf ball (if B) intersected with the training range (never further out than the row already is)."""
    lo_r, hi_r = np.minimum(FEAT_LO[j], o), np.maximum(FEAT_HI[j], o)
    if B is None: return lo_r.astype(np.float32), hi_r.astype(np.float32)
    return np.maximum(o - np.float32(B), lo_r).astype(np.float32), np.minimum(o + np.float32(B), hi_r).astype(np.float32)

def candidates(xc, X0c, j, T, B, K):
    """candidate (row, value) moves for feature j: the K nearest thresholds on each side inside the allowed interval, plus the interval ends."""
    xv = xc[:, j]; a, b = interval(X0c[:, j], j, B); nT = len(T); idx = np.searchsorted(T, xv, side='right'); R, V = [], []
    for d in range(min(K, nT)):
        k = idx + d; v = T[np.minimum(k, nT - 1)]; ok = (k < nT) & (v <= b); R.append(np.flatnonzero(ok)); V.append(v[ok])
        k = idx - 1 - d; v = np.nextafter(T[np.maximum(k, 0)], np.float32(-np.inf)); ok = (k >= 0) & (v >= a); R.append(np.flatnonzero(ok)); V.append(v[ok])
    ok = b > xv; R.append(np.flatnonzero(ok)); V.append(b[ok]); ok = a < xv; R.append(np.flatnonzero(ok)); V.append(a[ok])
    return np.concatenate(R), np.concatenate(V).astype(np.float32)

def one_step(margin, xc, X0c, g, feats, TH, B, K):
    """best single-feature move per row (by the model's own margin gap). Returns best_gap, best_feature (-1 = none helps), best_value, n_scored."""
    n = len(xc); best_g = g.copy(); best_j = np.full(n, -1, np.int64); best_v = np.zeros(n, np.float32); n_scored = 0
    bufR, bufJ, bufV = [], [], []; size = 0
    def flush():
        nonlocal n_scored
        if not bufR: return
        R, J, V = np.concatenate(bufR), np.concatenate(bufJ), np.concatenate(bufV); bufR.clear(); bufJ.clear(); bufV.clear()
        for s in range(0, len(R), BLOCK):
            r, jj, v = R[s:s + BLOCK], J[s:s + BLOCK], V[s:s + BLOCK]
            Xc = xc[r]; Xc[np.arange(len(r)), jj] = v                    # xc[r] is a copy
            gg = gap_of(margin(Xc)); n_scored += len(r)
            o = np.argsort(r, kind='stable'); rs, gs = r[o], gg[o]
            starts = np.flatnonzero(np.r_[True, rs[1:] != rs[:-1]]); segmax = np.maximum.reduceat(gs, starts); cnt = np.diff(np.r_[starts, len(rs)])
            hit = np.flatnonzero(gs == np.repeat(segmax, cnt)); _, first = np.unique(rs[hit], return_index=True); sel = o[hit[first]]
            rows = r[sel]; better = gg[sel] > best_g[rows]; rr = rows[better]
            best_g[rr] = gg[sel][better]; best_j[rr] = jj[sel][better]; best_v[rr] = v[sel][better]
    for j in feats:
        T = TH.get(int(j))
        if T is None: continue
        R, V = candidates(xc, X0c, int(j), T, B, K)
        if len(R) == 0: continue
        bufR.append(R); bufJ.append(np.full(len(R), j, np.int64)); bufV.append(V); size += len(R)
        if size >= BLOCK: flush(); size = 0
    flush(); return best_g, best_j, best_v, n_scored

def whitebox_trace(margin, X0, y0, feats, TH, B, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):
    """as NB3c's whitebox, plus a trace: per round, the evaded share per class (classes 4, 5) and the median margin gap of the rows not yet evaded."""
    x = X0.copy(); g = gap_of(margin(x)); active = g <= 0; stalled = np.zeros(len(x), bool); trace = []; t_start = time.time()
    active = ~(g > 0); init_ev = g > 0
    for it in range(max_iters):
        act = np.flatnonzero(active)
        if len(act) == 0: break
        t0 = time.time(); scored = 0
        for s in range(0, len(act), chunk):
            ids = act[s:s + chunk]; bg, bj, bv, ns = one_step(margin, x[ids], X0[ids], g[ids], feats, TH, B, K); scored += ns
            imp = bj >= 0; ii = ids[imp]; x[ii, bj[imp]] = bv[imp]; g[ii] = bg[imp]; stalled[ids[~imp]] = True; active[ids[~imp]] = False
        active &= (g <= 0); ev = g > 0
        trace.append({'round': it + 1, 'active': int(len(act)), 'evaded_c4': float(ev[y0 == 4].mean()), 'evaded_c5': float(ev[y0 == 5].mean()),
                      'median_gap_unevaded': float(np.median(g[~ev])) if (~ev).any() else float('nan'), 'seconds': time.time() - t0})
        if verbose and ((it + 1) % 10 == 0 or it == 0): print(f'      round {it + 1:>3}: active {len(act):>5} | evaded ransomware {trace[-1]["evaded_c4"]:.3f} reverseshell {trace[-1]["evaded_c5"]:.3f} | median gap of unevaded {trace[-1]["median_gap_unevaded"]:+.3f} | {time.time() - t0:4.1f}s', flush=True)
    return x, g, init_ev, stalled, trace
def whitebox(margin, X0, feats, TH, B, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):      # used by the self-test
    return whitebox_trace(margin, X0, np.zeros(len(X0), dtype=np.int64), feats, TH, B, K, max_iters, chunk, verbose)
print('white-box attack defined')

In [ ]:
def selftest(clf, margin, TH, seed=0):
    rng = np.random.RandomState(seed); rows = rng.choice(len(X_atk_test), 60, replace=False); X0 = X_atk_test[rows]; feats = rng.choice(CYBER_IDX, 12, replace=False)
    g0 = gap_of(margin(X0)); worst = 0.0
    for B in [0.1, 0.5, None]:
        bg, bj, bv, _ = one_step(margin, X0.copy(), X0, g0, feats, TH, B, K=10 ** 6)
        for i in range(len(rows)):
            best = g0[i]
            for j in feats:
                T = TH.get(int(j))
                if T is None: continue
                a, b = interval(X0[i:i + 1, j], int(j), B); a, b = float(a[0]), float(b[0]); inner = T[(T > a) & (T < b)]
                pts = np.concatenate([[a, b], (inner[:-1] + inner[1:]) / 2 if len(inner) > 1 else [], inner, np.nextafter(inner, np.float32(-np.inf))]).astype(np.float32)
                pts = pts[(pts >= a) & (pts <= b)]
                Xc = np.repeat(X0[i:i + 1], len(pts), 0); Xc[:, j] = pts; best = max(best, gap_of(margin(Xc)).max())
            worst = max(worst, best - bg[i])
            assert bg[i] >= best - 1e-4, f'one_step missed a better move (row {i}, budget {B}): {bg[i]} < {best}'
    # constraints after a short attack
    sub = X_atk_test[rng.choice(len(X_atk_test), 400, replace=False)]
    for B in [0.1, 0.5]:
        xa, ga, ie, st, lg = whitebox(margin, sub, CYBER_IDX, TH, B, max_iters=4, verbose=False)
        d = np.abs(xa - sub); assert d[:, CYBER_IDX].max() <= B + 1e-6, 'L-inf budget violated'
        other = np.setdiff1d(np.arange(N_FEAT), CYBER_IDX); assert d[:, other].max() == 0, 'a non-controllable feature changed'
        lo_ok = (xa >= np.minimum(FEAT_LO, sub) - 1e-6) & (xa <= np.maximum(FEAT_HI, sub) + 1e-6); assert lo_ok.all(), 'left the training range'
        assert (ga >= gap_of(margin(sub)) - 1e-9).all(), 'margin gap decreased'
    print(f'self-test passed (largest shortfall of one_step vs full enumeration: {max(worst, 0):.2e}); budget, range, non-controllable features and monotone margin all hold')

## 5. Surrogate and PGD (NB3 code)

In [ ]:
class SurrogateMLP(nn.Module):
    def __init__(self, n_features, n_classes=6, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_features, hidden), nn.ReLU(), nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, n_classes))
    def forward(self, x): return self.net(x)
def train_surrogate(seed, X_tr, y_tr, n_epochs=20):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed); torch.cuda.manual_seed_all(seed)
    model = SurrogateMLP(X_tr.shape[1]).to(DEVICE); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    ds = TensorDataset(torch.from_numpy(X_tr).float(), torch.from_numpy(y_tr).long()); loader = DataLoader(ds, batch_size=256, shuffle=True, generator=torch.Generator().manual_seed(seed))
    for ep in range(n_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad(); F.cross_entropy(model(xb), yb).backward(); opt.step()
    model.eval(); return model
def pgd(model, X, y, eps=PGD_EPS, n_iter=PGD_ITERS, bs=65536):
    model.eval(); alpha = eps / 4; out = np.empty_like(X)
    for s in range(0, len(X), bs):
        yt = torch.from_numpy(y[s:s + bs]).long().to(DEVICE); X0 = torch.from_numpy(X[s:s + bs]).float().to(DEVICE); Xa = X0.clone()
        for _ in range(n_iter):
            Xa.requires_grad_(True); g = torch.autograd.grad(F.cross_entropy(model(Xa), yt), Xa)[0]; Xa = Xa.detach() + alpha * g.sign(); Xa = X0 + torch.clamp(Xa - X0, -eps, eps)
        out[s:s + bs] = Xa.detach().cpu().numpy()
    return out

def fit_pair(X, y, seed):
    """GPU-fitted classifier (used for the white-box margin search) and a CPU copy (the oracle for every reported number, as in NB4)."""
    clf = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); clf.fit(X, y, verbose=False)
    tmp = f'_tmp_xgb_{seed}_{os.getpid()}.json'; clf.save_model(tmp); cpu = xgb.XGBClassifier(); cpu.load_model(tmp); cpu.set_params(device='cpu', n_jobs=-1); os.remove(tmp); return clf, cpu

def ppo_train_examples(seed, oracle):
    rng = np.random.RandomState(seed + 77); Xs, ys, used = [], [], []
    for tag in TRAIN_PPO_TAGS:
        pat, b = SAVED_AGENTS[tag]; path = pat.format(v=PIPELINE_VERSION, s=seed) + '.zip'
        if not os.path.exists(path): print(f'    {path} not found -- skipped'); continue
        idx = np.concatenate([rng.choice(np.flatnonzero(y_atk_train == c), min(N_PPO_TRAIN_PER_CLASS, int((y_atk_train == c).sum())), replace=False) for c in range(1, 6)])
        res = run_batched(RecurrentPPO.load(path), X_atk_train[idx], oracle, b, mode='ppo'); Xs.append(res['x_final']); ys.append(y_atk_train[idx]); used.append((tag, float(res['evaded'].mean())))
    return (np.vstack(Xs), np.concatenate(ys), used) if Xs else (np.zeros((0, N_FEAT), np.float32), np.zeros(0, np.int64), used)

def row_keys(X): return {r.tobytes() for r in np.ascontiguousarray(X, dtype=np.float32)}
def pgd_eval(cpu_m, X_adv):
    pred = cpu_m.predict(X_adv); ev = pred[atk_te] == 0
    return {'per_class_evasion': per_class(ev), 'overall_evasion': float(ev.mean()), 'weighted_f1': float(f1_score(y_test, pred, average='weighted')), 'benign_recall': float((pred[y_test == 0] == 0).mean()), 'attack_detected': float((~ev).mean())}
print('defence components defined')

## 6. The autoencoder gate (Layer 2)

In [ ]:
class AE(nn.Module):
    def __init__(self, n_features, bottleneck=GATE_BOTTLENECK, hidden=GATE_HIDDEN):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_features, hidden), nn.GELU(), nn.Linear(hidden, bottleneck), nn.GELU())
        self.decoder = nn.Sequential(nn.Linear(bottleneck, hidden), nn.GELU(), nn.Linear(hidden, n_features))
    def forward(self, x): return self.decoder(self.encoder(x))
FW = torch.ones(N_FEAT, device=DEVICE); FW[torch.as_tensor(CYBER_IDX, device=DEVICE)] = GATE_CYBER_BOOST
def wmse(recon, target): return (((recon - target) ** 2) * FW).sum(1) / FW.sum()

class Gate:
    def __init__(self, models, tau): self.models = models; self.tau = float(tau)
    @torch.no_grad()
    def score(self, X, bs=65536):
        X = np.ascontiguousarray(X, dtype=np.float32); out = np.empty(len(X), np.float64)
        for s in range(0, len(X), bs):
            b = torch.from_numpy(X[s:s + bs]).to(DEVICE); out[s:s + bs] = torch.stack([wmse(m(b), b) for m in self.models]).mean(0).double().cpu().numpy()
        return out
    def flag(self, X): return self.score(X) > self.tau

def train_gate(seed, X_benign):
    from sklearn.model_selection import KFold
    oof = np.zeros(len(X_benign)); models = []
    for k, (tr, ho) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=seed).split(X_benign)):
        torch.manual_seed(seed * 10 + k); m = AE(N_FEAT).to(DEVICE); opt = torch.optim.Adam(m.parameters(), lr=1e-3)
        ds = TensorDataset(torch.from_numpy(X_benign[tr]).float()); loader = DataLoader(ds, batch_size=256, shuffle=True, generator=torch.Generator().manual_seed(seed * 10 + k))
        for ep in range(GATE_EPOCHS):
            m.train()
            for (xb,) in loader: xb = xb.to(DEVICE); opt.zero_grad(); wmse(m(xb), xb).mean().backward(); opt.step()
        m.eval(); models.append(m); oof[ho] = Gate([m], 0).score(X_benign[ho])
    return Gate(models, np.percentile(oof, GATE_PCTL)), oof

class PipelineOracle:
    """hard-label verdict of the whole pipeline: a row Layer 1 calls benign but the gate flags is reported as an attack (class 1)."""
    def __init__(self, clf, gate): self.clf = clf; self.gate = gate; self.n_queries = 0
    def predict(self, X):
        X = np.ascontiguousarray(X, dtype=np.float32); self.n_queries += len(X); p = self.clf.predict(X)
        return np.where((p == 0) & self.gate.flag(X), 1, p)

class PipelineMargin:
    """the quantity the fully adaptive white-box attacker maximises: min(classifier margin gap, GATE_SCALE * (tau - error) / tau); > 0 <=> evades both layers.
    Returned as a 2-column array so that the existing gap_of(z) = z[:,0] - max(z[:,1:]) equals this score."""
    def __init__(self, clf_gpu, gate): self.m = Margin(clf_gpu); self.gate = gate
    def __call__(self, X):
        s = np.minimum(gap_of(self.m(X)), GATE_SCALE * (self.gate.tau - self.gate.score(X)) / self.gate.tau)
        return np.stack([s, np.zeros_like(s)], 1)
print('gate, pipeline oracle and joint-score margin defined')

## 7. PPO agent trainer against the pipeline (NB4b code, resumable)

In [ ]:
def fp_of(*parts): return hashlib.sha256(repr(parts).encode()).hexdigest()[:16]
def model_fp(oracle, X_probe): return hashlib.sha256(np.asarray(oracle.predict(X_probe)).tobytes()).hexdigest()[:16]
def latest_ckpt(d):
    if not os.path.isdir(d): return None
    c = [f for f in os.listdir(d) if f.endswith('_steps.zip')]
    if not c: return None
    best = max(c, key=lambda f: int(f.rsplit('_steps.zip', 1)[0].rsplit('_', 1)[1])); return os.path.join(d, best)

def get_agent(variant, seed, oracle, mtag='rob'):      # variant = total budget (float)
    eps_total = float(variant); name = f'ppo_attack3_{mtag}_budget{str(variant).replace(".", "p")}_{PIPELINE_VERSION}_seed{seed}'   # no '.' in the name: SB3 only appends .zip when there is no extension
    fp = fp_of(model_fp(oracle, X_atk_train[:300]), EPS_BOUND, MAX_STEPS, EVASION_REWARD, STEP_PENALTY_SCALE, TIMEOUT_PENALTY, TOTAL_TIMESTEPS, eps_total, seed, N_CYBER, 'pipeline-gate', GATE_PCTL, round(oracle.gate.tau, 8))
    if os.path.exists(name + '.zip') and os.path.exists(name + '.fp') and open(name + '.fp').read() == fp:
        print(f'    [{variant}/{seed}] finished agent found (fingerprint matches) -- loaded'); return RecurrentPPO.load(name), 0.0
    ckdir = f'_ckpt_{name}'; os.makedirs(ckdir, exist_ok=True); fpf = os.path.join(ckdir, '_fp.txt')
    env = IDSEvasionEnv(X_atk_train, y_atk_train, oracle, seed=seed, eps_total_budget=eps_total)
    ck = latest_ckpt(ckdir)
    if ck and os.path.exists(fpf) and open(fpf).read() == fp:
        model = RecurrentPPO.load(ck, env=env); remaining = TOTAL_TIMESTEPS - model.num_timesteps
        print(f'    [{variant}/{seed}] resuming from {model.num_timesteps}/{TOTAL_TIMESTEPS}')
    else:
        shutil.rmtree(ckdir, ignore_errors=True); os.makedirs(ckdir); open(fpf, 'w').write(fp)
        model = RecurrentPPO('MlpLstmPolicy', env, n_steps=128, batch_size=64, learning_rate=3e-4, gamma=0.99, seed=seed, verbose=0,
                             policy_kwargs=dict(lstm_hidden_size=64, n_lstm_layers=1), device='cpu'); remaining = TOTAL_TIMESTEPS
    t0 = time.time()
    if remaining > 0:
        model.learn(total_timesteps=remaining, reset_num_timesteps=False, callback=CheckpointCallback(save_freq=CKPT_FREQ, save_path=ckdir, name_prefix='ckpt'), progress_bar=False)
    model.save(name); open(name + '.fp', 'w').write(fp); shutil.rmtree(ckdir, ignore_errors=True)
    return model, time.time() - t0

## 8. Test rows

In [ ]:
rng0 = np.random.RandomState(0); pick = np.sort(np.concatenate([rng0.choice(np.flatnonzero(y_atk_test == c), N_WB_PER_CLASS, replace=False) for c in range(1, 6)]))
X_wb, y_wb = X_atk_test[pick], y_atk_test[pick]                           # NB5 / NB5b / NB5c rows
rng1 = np.random.RandomState(1); sub = np.sort(np.concatenate([rng1.choice(np.flatnonzero(y_wb == c), N_AW_PER_CLASS, replace=False) for c in range(1, 6)]))   # random subset (not the earliest rows)
X_aw, y_aw = X_wb[sub], y_wb[sub]
def pcs(ev, y): return np.array([ev[y == c].mean() for c in range(1, 6)])
def load_model(path, seed):
    gpu = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); gpu.load_model(path)
    cpu = xgb.XGBClassifier(); cpu.load_model(path); cpu.set_params(device='cpu', n_jobs=-1); return gpu, cpu
print(f'attack rows used by the white-box attackers: {len(X_aw)} ({N_AW_PER_CLASS} per class) | budgets {AW_BUDGETS} | PGD / PPO use all {len(X_atk_test):,} test attack rows')

## 9. Per-seed run

In [ ]:
def verdict(clf_cpu, gate, X, y):
    """per-row outcome for attack rows X: Layer-1 evasion, gate flag, pipeline evasion."""
    ev1 = clf_cpu.predict(X) == 0; fl = gate.flag(X); return ev1, fl, ev1 & ~fl
def pack(ev1, fl, evp, y):
    return {'l1_evasion': float(ev1.mean()), 'gate_detects_all': float(fl.mean()), 'gate_catches_of_l1_evaded': float(fl[ev1].mean()) if ev1.any() else float('nan'), 'pipeline_evasion': float(evp.mean()),
            'l1_pc': pcs(ev1, y), 'pipeline_pc': pcs(evp, y), 'gate_pc': pcs(fl, y)}

R = {}; t_all = time.time(); BIG = {}
for seed in SEEDS:
    pf = f'attack3_seed{seed}_{PIPELINE_VERSION}.pkl'
    cfg_fp = repr((best_params, LAYER1, AW_BUDGETS, N_AW_PER_CLASS, MAX_ITERS_AWARE, GATE_SCALE, N_FOLDS, GATE_EPOCHS, GATE_PCTL, GATE_HIDDEN, GATE_BOTTLENECK, GATE_CYBER_BOOST, RUN_PPO_AWARE, PPO_AWARE_MODELS, PPO_AWARE_BUDGETS, PPO_UNAWARE_TAGS, K_CAND, seed, X_atk_test.shape, float(X_atk_test[:5].sum()), 'nb6-v1'))
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') == cfg_fp: R[seed] = saved['R']; print(f'seed {seed}: result file found -- skipped'); continue
    t0 = time.time(); print(f'=== seed {seed}', flush=True); r = {}
    # ---- Layer 2: gate (benign training rows only) ----
    gate, oof = train_gate(seed, X_train[y_train == 0]); torch.save({'state': [m.state_dict() for m in gate.models], 'tau': gate.tau, 'cfg': dict(hidden=GATE_HIDDEN, bottleneck=GATE_BOTTLENECK, pctl=GATE_PCTL, boost=GATE_CYBER_BOOST)}, f'ae_gate_{PIPELINE_VERSION}_seed{seed}.pt')
    sc_te = gate.score(X_test); fpr = float((sc_te[y_test == 0] > gate.tau).mean()); tpr = float((sc_te[atk_te] > gate.tau).mean())
    r['gate_clean'] = {'tau': gate.tau, 'oof_mean': float(oof.mean()), 'fpr_test': fpr, 'tpr_test': tpr, 'tpr_pc': np.array([(sc_te[y_test == c] > gate.tau).mean() for c in range(1, 6)]), 'auc': float(__import__('sklearn.metrics', fromlist=['roc_auc_score']).roc_auc_score(y_test != 0, sc_te))}
    print(f'  gate: tau {gate.tau:.5f} | clean test FPR {fpr:.3f} TPR {tpr:.3f} AUC {r["gate_clean"]["auc"]:.3f} | per-class TPR {np.round(r["gate_clean"]["tpr_pc"], 3).tolist()}', flush=True)
    # ---- PGD (NB3 surrogate) examples, once ----
    sp = f'surrogate_mlp_{PIPELINE_VERSION}_seed{seed}.pt'; X_pgd = None
    if os.path.exists(sp):
        sur = SurrogateMLP(N_FEAT).to(DEVICE); sur.load_state_dict(torch.load(sp, map_location=DEVICE)); sur.eval(); X_pgd = pgd(sur, X_test, y_test)[atk_te]
        r['gate_vs_pgd'] = float(gate.flag(X_pgd).mean()); print(f'  gate alone flags {r["gate_vs_pgd"]:.3f} of PGD attack rows (clean attack rows: {tpr:.3f})', flush=True)
    for mtag, pat in LAYER1.items():
        mpath = pat.format(v=PIPELINE_VERSION, s=seed)
        if not os.path.exists(mpath): print(f'  {mpath} missing -- skipped'); continue
        gpu_m, cpu_m = load_model(mpath, seed); mg = Margin(gpu_m); TH = get_thresholds(gpu_m)
        if seed == SEEDS[0] and mtag == 'rob':
            selftest(gpu_m, mg, TH); pm = PipelineMargin(gpu_m, gate); ii = np.random.RandomState(0).choice(len(X_aw), min(2000, len(X_aw)), replace=False); a = pm(X_aw[ii])[:, 0] > 0; b = verdict(cpu_m, gate, X_aw[ii], y_aw[ii])[2]
            print(f'  joint-score sign vs CPU classifier + gate verdict: {(a == b).mean():.4f} agreement on {len(ii):,} attack rows'); assert (a == b).mean() > 0.99
        cl = cpu_m.predict(X_test); clean_pass = float(((cl == 0) & ~gate.flag(X_test))[y_test == 0].mean()); clean_l1_pass = float((cl[y_test == 0] == 0).mean()); r[f'{mtag}|clean'] = {'benign_pass_layer1': clean_l1_pass, 'benign_pass_pipeline': clean_pass, 'attack_detect_layer1': float((cl[atk_te] != 0).mean()), 'attack_detect_pipeline': float(1 - ((cl == 0) & ~gate.flag(X_test))[atk_te].mean())}
        print(f'  [{mtag}] clean: benign passes Layer 1 {clean_l1_pass:.3f}, passes pipeline {clean_pass:.3f}', flush=True)
        # 1. non-adaptive attackers
        if X_pgd is not None: r[f'{mtag}|pgd'] = pack(*verdict(cpu_m, gate, X_pgd, y_atk_test), y_atk_test)
        for tag in PPO_UNAWARE_TAGS:
            pat_, b = SAVED_AGENTS[tag]; path = pat_.format(v=PIPELINE_VERSION, s=seed) + '.zip'
            if not os.path.exists(path): continue
            res = run_batched(RecurrentPPO.load(path), X_atk_test, Oracle(cpu_m), b, mode='ppo'); xf = res['x_final']; ev1 = res['pred_final'] == 0; fl = gate.flag(xf); r[f'{mtag}|ppo_unaware_{tag}'] = pack(ev1, fl, ev1 & ~fl, y_atk_test)
        # 2. Layer-1-adaptive, gate-unaware white-box   3. fully adaptive white-box
        for bt, B in AW_BUDGETS.items():
            ts = time.time(); xa = whitebox_trace(mg, X_aw, y_aw, CYBER_IDX, TH, B, verbose=False)[0]; r[f'{mtag}|wb_unaware_{bt}'] = pack(*verdict(cpu_m, gate, xa, y_aw), y_aw); t1 = (time.time() - ts) / 60
            ts = time.time(); xa, ga, ie, st, tr = whitebox_trace(PipelineMargin(gpu_m, gate), X_aw, y_aw, CYBER_IDX, TH, B, max_iters=MAX_ITERS_AWARE, verbose=False); ev1, fl, evp = verdict(cpu_m, gate, xa, y_aw)
            r[f'{mtag}|wb_aware_{bt}'] = {**pack(ev1, fl, evp, y_aw), 'still_improving': float((~st & ~evp).mean()), 'stalled': float((st & ~evp).mean()), 'search_vs_cpu_disagree': float(((ga > 0) != evp).mean()), 'rounds': len(tr)}
            u, a_ = r[f'{mtag}|wb_unaware_{bt}'], r[f'{mtag}|wb_aware_{bt}']
            print(f'  [{mtag}] white-box {bt} sigma: Layer 1 only {u["l1_evasion"]:.3f} | gate-unaware -> pipeline {u["pipeline_evasion"]:.3f} (gate catches {u["gate_catches_of_l1_evaded"]:.2f} of Layer-1 evasions) | FULLY ADAPTIVE -> pipeline {a_["pipeline_evasion"]:.3f} per-class {np.round(a_["pipeline_pc"], 3).tolist()} (still improving {a_["still_improving"]:.3f}, rounds {a_["rounds"]}) | {t1:.1f}+{(time.time() - ts) / 60:.1f} min', flush=True)
        # 4. fully adaptive black-box PPO
        if RUN_PPO_AWARE and mtag in PPO_AWARE_MODELS:
            pipe = PipelineOracle(cpu_m, gate)
            for bt, B in PPO_AWARE_BUDGETS.items():
                agent, secs = get_agent(B, seed, pipe, mtag); res = run_batched(agent, X_atk_test, pipe, B, mode='ppo'); xf = res['x_final']; evp = res['pred_final'] == 0
                ev1 = cpu_m.predict(xf) == 0; fl = gate.flag(xf); r[f'{mtag}|ppo_aware_{bt}'] = {**pack(ev1, fl, evp, y_atk_test), 'train_min': secs / 60}
                print(f'  [{mtag}] PPO trained against the pipeline, {bt} sigma: pipeline evasion {evp.mean():.3f} per-class {np.round(pcs(evp, y_atk_test), 3).tolist()} (gate-unaware agent same budget: {r.get(f"{mtag}|ppo_unaware_{bt}", {}).get("pipeline_evasion", float("nan")):.3f}) | train {secs / 60:.1f} min', flush=True)
        del mg, gpu_m, cpu_m; torch.cuda.empty_cache()
    R[seed] = r; joblib.dump({'fp': cfg_fp, 'R': r}, pf); print(f'  seed {seed} done in {(time.time() - t0) / 60:.1f} min', flush=True)
print(f'Total {(time.time() - t_all) / 60:.1f} min')

## 10. Results

In [ ]:
atk_names = class_names[1:]
def ptest(a, b):
    d = np.asarray(a) - np.asarray(b)
    if len(d) < 2: return float('nan')
    return 1.0 if d.std(ddof=1) == 0 and d.mean() == 0 else (0.0 if d.std(ddof=1) == 0 else float(stats.ttest_rel(a, b).pvalue))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rk, i in enumerate(o): run = max(run, (len(p) - rk) * p[i]); adj[i] = min(1.0, run)
    return adj
ms = lambda x: f'{np.mean(x):.3f} +/- {(np.std(x, ddof=1) if len(x) > 1 else 0):.3f}'
V = lambda key, f: np.array([R[s][key][f] for s in SEEDS if key in R[s]])
print('10a. The gate on its own (clean test data; threshold = 95th pct of out-of-fold benign error)')
g = [R[s]['gate_clean'] for s in SEEDS]; print(f'   false-positive rate on benign {ms([x["fpr_test"] for x in g])} | detection of attack rows {ms([x["tpr_test"] for x in g])} | AUC {ms([x["auc"] for x in g])}')
print('   per-class detection:', {c: ms([x["tpr_pc"][i] for x in g]) for i, c in enumerate(atk_names)})
if all('gate_vs_pgd' in R[s] for s in SEEDS): print(f'   gate alone flags {ms([R[s]["gate_vs_pgd"] for s in SEEDS])} of PGD (NB3 surrogate) attack rows')
for m in LAYER1:
    if all(f'{m}|clean' in R[s] for s in SEEDS): c = [R[s][f'{m}|clean'] for s in SEEDS]; print(f'   [{m}] clean benign passing: Layer 1 {ms([x["benign_pass_layer1"] for x in c])} -> pipeline {ms([x["benign_pass_pipeline"] for x in c])} | clean attack detection: Layer 1 {ms([x["attack_detect_layer1"] for x in c])} -> pipeline {ms([x["attack_detect_pipeline"] for x in c])}')
rows = []
def add(label, keyfn, m):
    ks = [k for k in [keyfn] if all(f'{m}|{k}' in R[s] for s in SEEDS)]
    if not ks: return
    k = f'{m}|{ks[0]}'; l1, pe, gc = V(k, 'l1_evasion'), V(k, 'pipeline_evasion'), V(k, 'gate_catches_of_l1_evaded')
    rows.append({'Layer 1': m, 'attacker': label, 'evades Layer 1': ms(l1), 'evades pipeline (L1 + gate)': ms(pe), 'gate catches (of L1 evasions)': ms(gc[~np.isnan(gc)]) if (~np.isnan(gc)).any() else 'n/a', 'p (pipeline vs L1)': f'{ptest(pe, l1):.4f}'})
for m in LAYER1:
    add('PGD, NB3 surrogate (non-adaptive)', 'pgd', m)
    for t in PPO_UNAWARE_TAGS: add(f'PPO {t} sigma, trained vs standard classifier (non-adaptive)', f'ppo_unaware_{t}', m)
    for bt in AW_BUDGETS: add(f'white-box {bt} sigma, gate-UNAWARE', f'wb_unaware_{bt}', m); add(f'white-box {bt} sigma, FULLY ADAPTIVE', f'wb_aware_{bt}', m)
    for bt in PPO_AWARE_BUDGETS: add(f'PPO {bt} sigma, FULLY ADAPTIVE (trained vs pipeline)', f'ppo_aware_{bt}', m)
tab = pd.DataFrame(rows); pd.set_option('display.width', 340); pd.set_option('display.max_colwidth', 80); print('\n10b. Evasion of attack rows (mean +/- std over seeds)'); print(tab.to_string(index=False))
print('\n10c. Does knowing the gate help the attacker? (pipeline evasion, fully adaptive minus gate-unaware, paired over seeds)')
for m in LAYER1:
    for bt in AW_BUDGETS:
        ku, ka = f'{m}|wb_unaware_{bt}', f'{m}|wb_aware_{bt}'
        if all(ku in R[s] and ka in R[s] for s in SEEDS): u, a = V(ku, 'pipeline_evasion'), V(ka, 'pipeline_evasion'); print(f'   [{m}] white-box {bt} sigma: unaware {u.mean():.3f} -> fully adaptive {a.mean():.3f} (change {(a - u).mean():+.3f}, p {ptest(a, u):.4f}) | Layer-1-only evasion {V(ku, "l1_evasion").mean():.3f}')
    for bt in PPO_AWARE_BUDGETS:
        ku, ka = f'{m}|ppo_unaware_{bt}', f'{m}|ppo_aware_{bt}'
        if all(ku in R[s] and ka in R[s] for s in SEEDS): u, a = V(ku, 'pipeline_evasion'), V(ka, 'pipeline_evasion'); print(f'   [{m}] PPO {bt} sigma: unaware {u.mean():.3f} -> trained against the pipeline {a.mean():.3f} (change {(a - u).mean():+.3f}, p {ptest(a, u):.4f})')
print('\n10d. Per-class pipeline evasion, fully adaptive white-box (mean over seeds)')
pcr = []
for m in LAYER1:
    for bt in AW_BUDGETS:
        k = f'{m}|wb_aware_{bt}'
        if all(k in R[s] for s in SEEDS):
            a = np.stack([R[s][k]['pipeline_pc'] for s in SEEDS]); l1 = np.stack([R[s][k]['l1_pc'] for s in SEEDS]); u = np.stack([R[s][f'{m}|wb_unaware_{bt}']['pipeline_pc'] for s in SEEDS]); adj = holm(np.nan_to_num([ptest(a[:, i], u[:, i]) for i in range(5)], nan=1.0))
            pcr.append({'Layer 1': m, 'budget': f'{bt} sigma', **{c: f'{u[:, i].mean():.3f} -> {a[:, i].mean():.3f}' + ('*' if adj[i] < 0.05 else '') for i, c in enumerate(atk_names)}})
pc_tab = pd.DataFrame(pcr); print('   (gate-unaware pipeline evasion -> fully adaptive pipeline evasion; * = Holm p < 0.05)'); print(pc_tab.to_string(index=False))
print('\n10e. Search health of the fully adaptive white-box: rows still improving at the round limit, and search/CPU disagreement')
for m in LAYER1:
    for bt in AW_BUDGETS:
        k = f'{m}|wb_aware_{bt}'
        if all(k in R[s] for s in SEEDS): print(f'   [{m}] {bt} sigma: still improving {V(k, "still_improving").mean():.3f} | stalled {V(k, "stalled").mean():.3f} | disagreement {V(k, "search_vs_cpu_disagree").mean():.5f}')

## 11. Figure and save

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5)); width = 0.2
for mi, m in enumerate(LAYER1):
    keys = [(f'wb_unaware_{bt}', f'wb_aware_{bt}') for bt in AW_BUDGETS if all(f'{m}|wb_aware_{bt}' in R[s] for s in SEEDS)]
    if not keys: continue
    xs = np.arange(len(keys)); a = ax[mi]
    for off, (lab, fn) in enumerate([('Layer 1 only (gate-unaware)', lambda k: V(f'{m}|{k[0]}', 'l1_evasion')), ('pipeline, gate-unaware attacker', lambda k: V(f'{m}|{k[0]}', 'pipeline_evasion')), ('pipeline, fully adaptive attacker', lambda k: V(f'{m}|{k[1]}', 'pipeline_evasion'))]):
        v = [fn(k) for k in keys]; a.bar(xs + (off - 1) * width * 1.3, [x.mean() for x in v], width * 1.3, yerr=[x.std(ddof=1) if len(x) > 1 else 0 for x in v], capsize=2, label=lab)
    a.set_xticks(xs); a.set_xticklabels([f'{k[0].split("_")[-1]} sigma' for k in keys]); a.set_ylabel('evasion of attack rows'); a.set_title(f'White-box attacker vs Layer 1 = {m}'); a.legend(fontsize=8); a.grid(alpha=.3, axis='y'); a.set_ylim(0, 1.05)
plt.tight_layout(); plt.savefig(f'attack3_adaptive_gate_{PIPELINE_VERSION}.png', dpi=150); plt.show()
joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'table': tab, 'per_class_table': pc_tab, 'per_seed': {s: R[s] for s in SEEDS}, 'config': {'AW_BUDGETS': AW_BUDGETS, 'N_AW_PER_CLASS': N_AW_PER_CLASS, 'MAX_ITERS_AWARE': MAX_ITERS_AWARE, 'GATE_SCALE': GATE_SCALE, 'GATE_PCTL': GATE_PCTL, 'PPO_AWARE_BUDGETS': PPO_AWARE_BUDGETS}}, f'attack3_results_{PIPELINE_VERSION}.pkl')
tab.to_csv(f'attack3_table_{PIPELINE_VERSION}.csv', index=False); pc_tab.to_csv(f'attack3_per_class_{PIPELINE_VERSION}.csv', index=False)
json.dump({'notebook': 'NB6', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'xgboost': xgb.__version__, 'torch': torch.__version__, 'sb3_contrib': sb3_contrib.__version__, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb6_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'attack3_results_{PIPELINE_VERSION}.pkl', f'attack3_table_{PIPELINE_VERSION}.csv', f'attack3_per_class_{PIPELINE_VERSION}.csv', f'attack3_adaptive_gate_{PIPELINE_VERSION}.png', f'nb6_run_manifest_{PIPELINE_VERSION}.json'] + [f'ae_gate_{PIPELINE_VERSION}_seed{s}.pt' for s in SEEDS]: print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)